# PokerMind AI — Analyse approfondie du modèle préflop (Notebook 03)

## 1. Présentation du notebook

### Objectif

Ce notebook approfondit l'analyse du **modèle préflop de référence** identifié dans le Notebook 02. Son but n'est pas d'entraîner de nouveaux modèles, mais de **comprendre ce que le modèle a réellement appris** et d'évaluer honnêtement ses performances et ses limites.

### Modèle analysé

Le modèle de référence est une **Régression Logistique avec pondération des classes** (`class_weight='balanced'`), entraîné sur les variables préflop uniquement. Ce choix est justifié par :
- les meilleures performances en AUC-ROC parmi les modèles préflop du Notebook 02
- une meilleure interprétabilité directe (coefficients lisibles)
- une calibration probabiliste naturellement plus stable que les modèles à base d'arbres

### Variables utilisées

Uniquement les **variables préflop** : cartes initiales, position, blindes, taille de pile. Aucune variable d'action n'est utilisée.

### Plan du notebook

1. Chargement des données et entraînement du modèle de référence
2. Calibration — les probabilités prédites sont-elles fiables ?
3. Courbe ROC — discrimination globale du modèle
4. Analyse des erreurs — quels profils le modèle prédit-il mal ?
5. Variables à importance nulle — interprétation structurelle
6. Coefficients — ce que le modèle a réellement appris
7. Discussion de généralisation — limites hors du contexte Pluribus
8. Conclusions académiques

## 2. Imports et configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..")
PLAYER_FEATURE_FILE = PROJECT_ROOT / "data" / "processed" / "player_level_features.csv"

## 3. Chargement des données et entraînement du modèle de référence

On recharge le jeu de données joueur et on entraîne la Régression Logistique sur **l'ensemble des données**. Ce modèle complet servira à l'analyse des coefficients.

Pour la calibration et l'analyse des erreurs, on utilisera des **prédictions hors-pli** (*out-of-fold*) via `cross_val_predict` avec une validation croisée à 5 plis. Cela garantit que chaque prédiction a été produite par un modèle qui n'avait pas vu la ligne correspondante lors de l'entraînement — ce qui donne une estimation honnête des probabilités sur l'ensemble du jeu de données.

In [ ]:
print(f"Répertoire de travail actuel : {Path('.').resolve()}")
print(f"Chemin absolu du fichier attendu : {PLAYER_FEATURE_FILE.resolve()}")

if not PLAYER_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {PLAYER_FEATURE_FILE.resolve()}\n"
        "Vérifiez que le Notebook 02 a été exécuté et a généré 'player_level_features.csv'."
    )
if PLAYER_FEATURE_FILE.stat().st_size < 1024:
    raise ValueError(
        f"Le fichier semble vide ou corrompu ({PLAYER_FEATURE_FILE.stat().st_size} octets).\n"
        "Régénérez le fichier en exécutant le Notebook 02 depuis le début."
    )

df = pd.read_csv(PLAYER_FEATURE_FILE)

required_columns = ["player_won", "preflop_strength_score", "starting_stack", "player_num_folds"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le jeu de données : {missing_columns}")

PREFLOP_FEATURES = [
    "number_of_players",
    "starting_stack",
    "blind_or_straddle",
    "is_small_blind",
    "is_big_blind",
    "player_position_index",
    "is_pair",
    "is_suited",
    "high_card_rank",
    "low_card_rank",
    "rank_gap",
    "has_ace",
    "has_king",
    "preflop_strength_score",
    "variant",
]

target_column = "player_won"
X = df[PREFLOP_FEATURES].copy()
y = df[target_column].copy()

numeric_features = [f for f in PREFLOP_FEATURES if f != "variant"]
categorical_features = ["variant"]


def build_lr_pipeline():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
    ])


reference_model = build_lr_pipeline()
reference_model.fit(X, y)

print(f"Jeu de données : {X.shape[0]} lignes, {X.shape[1]} variables préflop — OK")
print(f"Distribution cible — perdant (0) : {(y == 0).sum()}, gagnant (1) : {(y == 1).sum()}")

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_proba = cross_val_predict(
    build_lr_pipeline(), X, y, cv=cv_strategy, method="predict_proba"
)[:, 1]

y_pred_class = (y_pred_proba >= 0.5).astype(int)

print(f"Prédictions hors-pli générées pour {len(y_pred_proba)} lignes (validation croisée, 5 plis).")

## 4. Calibration du modèle

La **courbe de calibration** compare la probabilité prédite par le modèle à la fréquence réelle de victoire observée dans les données. Un modèle parfaitement calibré suit la diagonale : si le modèle prédit 30 % de probabilité de gagner pour un groupe de joueurs, environ 30 % de ce groupe ont effectivement gagné.

**Ce qu'on cherche :**
- Courbe proche de la diagonale → les probabilités sont interprétables comme des fréquences réelles.
- Courbe systématiquement au-dessus → le modèle surestime les probabilités de victoire.
- Courbe en dessous → il les sous-estime.

**Note sur `class_weight='balanced'` :** la pondération des classes modifie le seuil de décision interne du modèle en le calibrant comme si les classes étaient équilibrées (50/50), alors que la réalité est 83/17. Cela peut décaler la courbe de calibration par rapport à la diagonale.

In [ ]:
prob_true, prob_pred = calibration_curve(y, y_pred_proba, n_bins=8)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(prob_pred, prob_true, "s-", label="Régression Logistique (préflop)", color="#2166ac")
ax.plot([0, 1], [0, 1], "k--", label="Calibration parfaite", linewidth=0.8)
ax.set_xlabel("Probabilité prédite (moyenne par groupe)")
ax.set_ylabel("Fraction de gagnants réels")
ax.set_title("Courbe de calibration — modèle préflop")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Observation — calibration

Avec `class_weight='balanced'`, la Régression Logistique est entraînée comme si les classes étaient équilibrées. Cela déplace le seuil de décision interne et produit généralement une **surestimation des probabilités de victoire** par rapport au taux de base réel (17 %).

**Ce que cela implique en pratique :** les probabilités produites par ce modèle ne doivent pas être interprétées comme des probabilités absolues directement comparables à un taux de victoire réel. Elles sont utiles pour **ordonner les joueurs** par probabilité relative (c'est ce que mesure l'AUC-ROC), mais pas comme des estimations de fréquence brute.

Pour un usage probabiliste absolu, une recalibration (régression isotonique ou Platt scaling) serait nécessaire. Cela dépasse la portée de ce baseline.

## 5. Courbe ROC

La **courbe ROC** (*Receiver Operating Characteristic*) représente le taux de vrais positifs (recall) en fonction du taux de faux positifs, pour tous les seuils de décision possibles. L'aire sous cette courbe (AUC-ROC) résume la capacité du modèle à discriminer les gagnants des perdants.

**Interprétation de l'AUC-ROC :**
- AUC = 0.5 → modèle aléatoire, aucune discrimination
- AUC = 1.0 → discrimination parfaite
- AUC > 0.8 → bonne discrimination, en particulier pour des variables préflop seulement

**Ce qu'on cherche :** confirmer que l'AUC obtenu en validation croisée est stable et significativement supérieur à 0.5.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y, y_pred_proba, ax=ax, name="Régression Logistique (préflop)"
)
ax.plot([0, 1], [0, 1], "k--", label="Modèle aléatoire")
ax.set_title("Courbe ROC — modèle préflop (validation croisée, 5 plis)")
ax.legend()
plt.tight_layout()
plt.show()

### Observation — courbe ROC

La courbe ROC confirme que le modèle préflop discrimine mieux que le hasard. L'AUC affiché est l'estimation hors-pli sur 5 plis, ce qui la rend fiable.

Ce résultat signifie que, si on choisit au hasard un gagnant et un perdant, le modèle attribue une probabilité plus élevée au gagnant dans une proportion correspondant à l'AUC. Sur des variables préflop seulement (cartes, position), ce signal est réel mais limité : la main de poker contient une composante aléatoire irréductible que les informations préflop seules ne peuvent pas capturer.

## 6. Analyse des erreurs

On examine les prédictions incorrectes du modèle pour comprendre **dans quels cas il se trompe** et si les erreurs ont des caractéristiques préflop particulières.

On distingue quatre catégories de prédictions :

| Catégorie | Prédit | Réel |
|---|---|---|
| **Vrai Positif** | gagnant | gagnant ✓ |
| **Vrai Négatif** | perdant | perdant ✓ |
| **Faux Positif** | gagnant | perdant ✗ |
| **Faux Négatif** | perdant | gagnant ✗ |

**Ce qu'on cherche :** les faux positifs et faux négatifs ont-ils des profils préflop caractéristiques ?

In [ ]:
error_type_map = {
    (0, 0): "Vrai Négatif",
    (1, 1): "Vrai Positif",
    (1, 0): "Faux Positif",
    (0, 1): "Faux Négatif",
}

df_analysis = df[PREFLOP_FEATURES + [target_column]].copy()
df_analysis["predicted_proba"] = y_pred_proba
df_analysis["predicted_class"] = y_pred_class
df_analysis["error_type"] = df_analysis.apply(
    lambda row: error_type_map[(row["predicted_class"], row[target_column])], axis=1
)

df_analysis["error_type"].value_counts().reset_index().rename(columns={"count": "nombre"})

In [ ]:
error_profile_df = df_analysis.groupby("error_type")[
    ["preflop_strength_score", "player_position_index", "is_pair", "is_suited", "predicted_proba"]
].mean().round(3)

error_profile_df

In [ ]:
error_order = ["Vrai Négatif", "Faux Positif", "Faux Négatif", "Vrai Positif"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df_analysis, x="error_type", y="preflop_strength_score",
    order=error_order, ax=axes[0]
)
axes[0].set_title("Score préflop par type de prédiction")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(
    data=df_analysis, x="error_type", y="predicted_proba",
    order=error_order, ax=axes[1]
)
axes[1].set_title("Probabilité prédite par type de prédiction")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

### Observation — analyse des erreurs

Le tableau et les graphiques révèlent le comportement typique du modèle sur ses erreurs :

**Faux Positifs** (prédit gagnant, effectivement perdant) :
- Ces joueurs ont un score préflop moyen **plus élevé** que les Vrais Négatifs.
- Le modèle a raison d'attribuer une probabilité élevée à des mains fortes — mais la main s'est mal terminée malgré de bonnes cartes initiales.
- Cela reflète l'**aléa irréductible** du poker : même la meilleure main préflop peut perdre selon le tirage du board ou les actions adverses.

**Faux Négatifs** (prédit perdant, effectivement gagnant) :
- Ces joueurs ont un score préflop moyen **plus faible** que les Vrais Positifs.
- Ils ont gagné malgré des cartes initialement défavorables — ce que le modèle ne peut pas anticiper à partir des seules informations préflop.

**Conclusion :** les erreurs ne sont pas des bugs du modèle — elles reflètent la **limite fondamentale de la prédiction préflop**. Le poker contient une composante aléatoire que les informations disponibles avant les décisions ne peuvent pas prédire parfaitement. Un modèle qui ferait moins d'erreurs à ce stade serait probablement en train d'exploiter de la fuite d'information.

## 7. Variables à importance nulle — interprétation structurelle

Dans le Notebook 02, trois variables avaient une importance nulle : `starting_stack`, `number_of_players`, et `variant`. Cette section confirme empiriquement pourquoi, et en tire les conséquences pour la généralisation du modèle.

**Ce qu'on cherche :** montrer que ces variables sont constantes dans le dataset Pluribus et expliquer ce que cela implique si on changeait de contexte.

In [ ]:
constant_features_check = pd.DataFrame({
    "variable": ["number_of_players", "starting_stack", "variant"],
    "valeurs_distinctes": [
        df["number_of_players"].nunique(),
        df["starting_stack"].nunique(),
        df["variant"].nunique(),
    ],
    "valeur(s) présente(s)": [
        str(sorted(df["number_of_players"].unique().tolist())),
        str(sorted(df["starting_stack"].unique().tolist())),
        str(sorted(df["variant"].unique().tolist())),
    ],
    "importance_arbre (notebook 02)": [0.0, 0.0, 0.0],
})

constant_features_check

### Observation — variables constantes dans Pluribus

Le tableau confirme que ces trois variables sont **constantes** dans le dataset :

- `number_of_players` = toujours **6**
- `starting_stack` = toujours **10 000**
- `variant` = toujours **NT** (No-Limit Texas Hold'em)

Une variable constante ne peut apporter aucune information discriminante — elle est identique pour les gagnants et les perdants. Son importance est donc mécaniquement nulle, quel que soit le modèle utilisé.

**Implication pour la généralisation :** ces trois variables pourraient être informatives dans un contexte de poker réel :
- `starting_stack` varie selon les joueurs et les parties (les stratégies optimales diffèrent selon si on est *short stack* ou *deep stack*)
- `number_of_players` varie (parties *heads-up*, 6-max, 9-max ont des dynamiques très différentes)
- `variant` varie selon les sites de jeu

Le modèle actuel est **spécifique au format Pluribus** et ne généralise pas naturellement à d'autres contextes.

## 8. Coefficients — ce que le modèle a réellement appris

La Régression Logistique produit des **coefficients** directement interprétables : un coefficient positif indique que la variable est associée à une plus grande probabilité de victoire, un coefficient négatif à une plus grande probabilité de défaite.

**Ce qu'on cherche :** vérifier que les variables apprises par le modèle sont cohérentes avec la théorie du poker, et identifier les variables dont le signal est quasi-nul.

In [ ]:
feature_names = reference_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = reference_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "variable": [name.replace("num__", "").replace("cat__", "") for name in feature_names],
    "coefficient": coefficients,
}).sort_values("coefficient", ascending=False).reset_index(drop=True)

coef_df

In [ ]:
plt.figure(figsize=(10, 6))
colors = ["#d73027" if c > 0 else "#4575b4" for c in coef_df["coefficient"]]
plt.barh(coef_df["variable"], coef_df["coefficient"], color=colors)
plt.axvline(x=0, color="black", linestyle="--", linewidth=0.8)
plt.title("Coefficients de la Régression Logistique — modèle préflop")
plt.xlabel("Coefficient (rouge → associé à la victoire, bleu → associé à la défaite)")
plt.tight_layout()
plt.show()

### Observation — ce que le modèle a appris

Les coefficients confirment et précisent ce que l'importance des variables de la forêt aléatoire suggérait dans le Notebook 02 :

**Variables positivement associées à la victoire (rouge) :**
- La **force des cartes** (`preflop_strength_score`, `high_card_rank`, `low_card_rank`) : plus les cartes initiales sont fortes, plus la probabilité de victoire augmente — cohérent avec la théorie du poker.
- La **paire** (`is_pair`) et les **cartes assorties** (`is_suited`) : avantage préflop reconnu par le modèle, cohérent avec les heuristiques classiques.
- L'**as** (`has_ace`) et le **roi** (`has_king`) : les cartes hautes contribuent positivement.

**Variables négativement associées (bleu) :**
- Un **grand écart entre les rangs** (`rank_gap`) : des cartes déconnectées réduisent les combinaisons de tirages possibles (straight, flush draw).
- La **petite blinde** (`is_small_blind`) peut présenter une légère association négative en raison de sa position désavantageuse en post-flop (première à agir).

**Variables à coefficient quasi-nul :**
- `number_of_players`, `starting_stack`, `variant` : constantes dans Pluribus, comme attendu.

**Ce que le modèle ne peut pas apprendre :**
Le modèle ne connaît pas les actions en cours de main, le tirage du board, ni le style de jeu des adversaires. Il extrait uniquement le signal statistique présent dans les cartes initiales et la position à la table.

## 9. Discussion de généralisation

Cette section examine comment le modèle préflop se comporterait si on l'appliquait à des données de poker réelles, différentes du dataset Pluribus.

### Le dataset Pluribus : un environnement contrôlé

Pluribus est un dataset de **recherche en IA**, ce qui le distingue fondamentalement du poker réel :

| Dimension | Pluribus | Poker réel (online) |
|---|---|---|
| Cartes des adversaires | Toutes visibles (log de recherche) | Cachées — inconnues |
| Taille de pile | Toujours 10 000 (fixe) | Variable (short stack, deep stack) |
| Nombre de joueurs | Toujours 6 | 2 à 9 selon la table |
| Variante | Toujours NT | Multiple (NL, PL, Limit…) |
| Profil des joueurs | Agents IA homogènes | Joueurs humains hétérogènes |

### Impact sur les variables du modèle

**Cartes adverses cachées :** dans des données réelles, les variables `card_1_rank`, `card_2_rank`, `is_pair`, `is_suited`, `preflop_strength_score`, etc. ne seraient disponibles que pour le héros. Pour les ~5/6 autres joueurs, ces variables seraient manquantes, ce qui réduirait drastiquement la qualité des prédictions pour la majorité des lignes.

**Taille de pile variable :** `starting_stack` deviendrait un signal informatif — les stratégies optimales diffèrent profondément selon qu'un joueur est *short stack* (< 20 blindes) ou *deep stack* (> 100 blindes).

**Nombre de joueurs variable :** la probabilité de victoire de base change mécaniquement avec le nombre de joueurs. `number_of_players` deviendrait une variable utile au modèle.

### Conclusion de généralisation

Le modèle actuel est **valide dans le contexte Pluribus** comme baseline académique. Il n'est **pas directement transférable** à des données de poker réelles sans modifications substantielles : recollection de données avec les variables manquantes, redéfinition des features, et réentraînement sur un dataset plus représentatif.

Cette limitation ne disqualifie pas l'approche méthodologique — elle en définit le périmètre d'application.

## 10. Conclusions académiques

### Ce que ce projet a démontré

Ce projet a construit et analysé un pipeline ML complet pour la prédiction de victoire au poker préflop, depuis les fichiers bruts PHH jusqu'à l'évaluation rigoureuse d'un modèle de référence.

### Ce que le modèle préflop fait bien

1. **Il capte un signal réel.** La force des cartes initiales est statistiquement corrélée avec la victoire, et le modèle l'exploite de façon cohérente avec la théorie du poker.

2. **Il est méthodologiquement honnête.** L'utilisation de `class_weight='balanced'`, de la validation croisée stratifiée, et de l'AUC-ROC comme métrique principale garantit des estimations solides malgré le déséquilibre de classes (83/17).

3. **Il démontre clairement la fuite d'information.** La comparaison avec le modèle main complète — dont 84 % de l'importance provient de `player_num_folds` — illustre de façon concrète pourquoi la séparation temporelle des variables est essentielle en ML sur données séquentielles.

### Limites consolidées

| Limite | Cause | Impact |
|---|---|---|
| Performance modeste | Aléa irréductible du poker | Attendu — pas une erreur |
| Position = numéro de siège | Bouton non parsé depuis les actions PHH | Signal de position imparfait |
| Probabilités non calibrées | `class_weight='balanced'` décale le seuil | Ne pas interpréter comme fréquences absolues |
| Généralisation limitée | Pluribus est un environnement contrôlé | Ne pas extrapoler directement |
| Proxy de victoire imparfait | `finishing_stack > starting_stack` | Mesure le gain, pas la qualité des décisions |
| Variables constantes inutilisables | Structure fixe de Pluribus | Réentraînement nécessaire sur données variées |

### Remarque finale

Les limites identifiées dans ce projet ne sont pas des échecs — elles sont des **observations méthodologiques nécessaires** pour circonscrire le périmètre de validité du modèle. Un baseline honnête et bien expliqué est plus valable académiquement qu'un modèle opaque avec de hautes performances inexpliquées.

### Prochaines étapes possibles

- Parser la position du bouton dans les actions PHH pour obtenir la vraie position de jeu (BTN, CO, HJ, UTG)
- Construire un jeu de données par rue (flop, turn, river) pour une analyse temporelle
- Développer une application Streamlit pour visualiser les prédictions main par main
- Explorer la recalibration du modèle pour un usage probabiliste absolu